# 🎬 ComfyUI + Wan2.1 動画生成（Google Colab T4対応）

Alibaba製の動画生成モデル **Wan2.1 1.3B** を Google Colab の T4 GPU で動かすための notebook です。

**実行順序:**
1. Environment Setup（ComfyUIインストール）
2. Step W1: モデルDL
3. Step W2: 動画保存先設定
4. Step W3: 確認
5. ComfyUI起動（cloudflared推奨）


In [ ]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  # ここをFalseにするだけで、100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install comfyui-workflow-templates
!pip install comfyui-embedded-docs

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


---
## 🎬 Wan2.1 動画生成セットアップ（T4対応・1.3B）

Alibaba製の動画生成モデル **Wan2.1 1.3B** をComfyUIで使うための設定です。

| 項目 | 内容 |
|------|------|
| モデル | Wan2.1 1.3B（テキスト→動画） |
| 必要VRAM | **8GB〜**（T4 16GBで余裕あり）|
| ライセンス | Apache 2.0（商用利用可）|
| 出力 | MP4動画 |


In [ ]:
# ============================================================
# Step W1: Wan2.1 モデルのダウンロード
# ============================================================
import os

# モデル保存先
diffusion_dir = "/content/ComfyUI/models/diffusion_models"
text_encoder_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
clip_vision_dir = "/content/ComfyUI/models/clip_vision"

for d in [diffusion_dir, text_encoder_dir, vae_dir, clip_vision_dir]:
    os.makedirs(d, exist_ok=True)

# --- メインモデル（1.3B・T2V）---
model_path = f"{diffusion_dir}/wan2.1_t2v_1.3B_bf16.safetensors"
if not os.path.exists(model_path):
    print("📥 Wan2.1 1.3B モデルをダウンロード中... (~6GB)")
    !wget -q --show-progress -c \
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors" \
        -P {diffusion_dir}
else:
    print("✅ メインモデルはすでにあります")

# --- テキストエンコーダ（umt5-xxl・量子化版）---
te_path = f"{text_encoder_dir}/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
if not os.path.exists(te_path):
    print("\n📥 テキストエンコーダ (UMT5-XXL fp8) をダウンロード中... (~5GB)")
    !wget -q --show-progress -c \
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors" \
        -P {text_encoder_dir}
else:
    print("\n✅ テキストエンコーダはすでにあります")

# --- VAE ---
vae_path = f"{vae_dir}/wan_2.1_vae.safetensors"
if not os.path.exists(vae_path):
    print("\n📥 VAE をダウンロード中...")
    !wget -q --show-progress -c \
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors" \
        -P {vae_dir}
else:
    print("\n✅ VAEはすでにあります")

print("\n✅ 全モデルのダウンロード完了")

In [ ]:
# ============================================================
# Step W1.5: ComfyUI-VideoHelperSuite のインストール
# （VHS_VideoCombine ノードでMP4出力するために必要）
# ============================================================
import os, subprocess, sys

vhs_dir = "/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite"

if not os.path.exists(vhs_dir):
    print("📥 ComfyUI-VideoHelperSuite をインストール中...")
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite {vhs_dir}
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         f"{vhs_dir}/requirements.txt"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ VideoHelperSuite インストール完了")
    else:
        print("⚠️ requirements.txt のインストールに失敗しました")
        print(result.stderr[-500:])
else:
    print("✅ VideoHelperSuite はすでにインストール済みです")
    %cd {vhs_dir}
    !git pull
    %cd /content/ComfyUI

# imageio-ffmpeg（MP4エンコードに必要）
!pip install -q imageio imageio-ffmpeg
print("✅ ffmpeg 依存関係のインストール完了")


In [ ]:
# ============================================================
# Step W2: 動画保存先をGoogleドライブにリンク
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

drive_video_path = "/content/drive/MyDrive/ComfyUI_Video"
os.makedirs(drive_video_path, exist_ok=True)

local_video_path = "/content/ComfyUI/output/video"
os.makedirs(os.path.dirname(local_video_path), exist_ok=True)

if not os.path.islink(local_video_path):
    if os.path.exists(local_video_path):
        import shutil
        shutil.rmtree(local_video_path)
    os.symlink(drive_video_path, local_video_path)

print(f"✅ 動画は {drive_video_path} に自動保存されます")

In [ ]:
# ============================================================
# Step W3: ダウンロード確認
# ============================================================
import os

checks = [
    ("Wan2.1 1.3B モデル",
     "/content/ComfyUI/models/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors"),
    ("テキストエンコーダ (UMT5-XXL fp8)",
     "/content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    ("VAE",
     "/content/ComfyUI/models/vae/wan_2.1_vae.safetensors"),
]

print("=" * 40)
print("  モデル確認")
print("=" * 40)
all_ok = True
for name, path in checks:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024 / 1024
        print(f"✅ {name}: {size:.1f} GB")
    else:
        print(f"❌ {name}: 見つかりません")
        all_ok = False

# VRAM確認
print()
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print("GPU:", r.stdout.strip())

print()
if all_ok:
    print("🎉 準備完了！ComfyUIを起動して、Wan2.1ワークフローをロードしてください。")
else:
    print("⚠️ Step W1 を再実行してください")

In [ ]:
!pip install av
!pip install comfy_aimdo

### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
# %cd /content/drive/MyDrive/ComfyUI
# !python main.py --dont-print-server

%cd /content/ComfyUI

# CPUで動かす場合はこちら（GPU制限解除待ちの場合）
# !python main.py --cpu --dont-print-server

# もしT4 GPUが使えるようになったら、--cpu を外して以下にしてください
!python main.py --dont-print-server

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server